In [1]:

# ---------- 0. Install (Kaggle / Colab) ----------
!pip install pennylane==0.35 autoray==0.6.9  --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.8/49.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 61.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 57.4 MB/s eta 0:00:00:00:01


In [2]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime 
!pip install qiskit-algorithms qiskit-optimization -q
from qiskit_algorithms.minimum_eigensolvers import QAOA, VQE, NumPyMinimumEigensolver
from qiskit_optimization.algorithms import MinimumEigenOptimizer, ADMMOptimizer
print("All imports OK")

from qiskit_ibm_runtime import QiskitRuntimeService
print("OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 57.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 79.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: PyJWT
    Found existing installation: PyJWT 2.10.1
    Uninstalling PyJWT-2.10.1:
      Successfully uninstalled PyJWT-2.10.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/samplomatic/__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


OK


In [3]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=secrets.get_secret("IBMTOKEN"),    # reads from Kaggle Secrets
    instance=secrets.get_secret("IBMCRN")   # reads from Kaggle Secrets
)

# Test it works:
print(service.backends())  # should print a list of quantum computers

# Check queue lengths — pick the one with fewest pending jobs
for b in service.backends():
    status = b.status()
    print(f"{b.name:20} | jobs in queue: {status.pending_jobs:>4} | operational: {status.operational}")

In [ ]:
!pip install pylatexenc

In [ ]:
# loading the best_model_quantum.pth 
import shutil

src = "/kaggle/input/datasets/tarunshr/epaper2/best_model_quantum.pth"
dst = "/kaggle/working/best_model_quantum.pth"

shutil.copy(src, dst)

print("✅ File copied successfully!")

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=secrets.get_secret("IBMTOKEN"),
    instance=secrets.get_secret("IBMCRN")
)

# Check your account usage
usage = service.usage()
print(usage)

In [ ]:

    import os, json, time, random, warnings
    import numpy as np
    import torch
    import torch.nn as nn
    import torchvision.transforms as transforms
    from torchvision.datasets import ImageFolder
    from torchvision.models import efficientnet_b0
    from torch.utils.data import DataLoader, Subset
    from collections import defaultdict
    
    import pennylane as qml
    
    from qiskit import QuantumCircuit as QiskitCircuit
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
    from qiskit.quantum_info import SparsePauliOp
    
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score,
        f1_score, confusion_matrix, roc_auc_score, classification_report
    )
    
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    import seaborn as sns
    
    from kaggle_secrets import UserSecretsClient
    
    warnings.filterwarnings("ignore")
    
    SEED = 42
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    
    print("✅ All imports OK")
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 1. CONFIG
    # ══════════════════════════════════════════════════════════════════════════════
    N_QUBITS  = 4
    Q_DEPTH   = 1
    SHOTS     = 1024
    BATCH_HW  = 10
    BEST_PATH = "/kaggle/working/best_model_quantum.pth"
    DATA_DIR  = "/kaggle/input/datasets/tarunshr/forest-fire/"
    OUT_DIR   = "/kaggle/working/hardware_eval"
    os.makedirs(OUT_DIR, exist_ok=True)
    device_cpu = torch.device("cpu")
    
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 2. REBUILD MODEL
    # ══════════════════════════════════════════════════════════════════════════════
    base_model = efficientnet_b0(weights="DEFAULT")
    base_model.classifier = nn.Identity()
    
    dev_pl = qml.device("default.qubit", wires=N_QUBITS)
    
    @qml.qnode(dev_pl, interface="torch")
    def quantum_circuit(inputs, weights):
        qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS))
        qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
        return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]
    
    class QuantumLayer(nn.Module):
        def __init__(self):
            super().__init__()
            self.qlayer = qml.qnn.TorchLayer(
                quantum_circuit, {"weights": (Q_DEPTH, N_QUBITS, 3)}
            )
        def forward(self, x):
            return self.qlayer(x)
    
    class EfficientNetQuantumClassifier(nn.Module):
        def __init__(self, n_classes=2):
            super().__init__()
            self.feature_extractor = base_model
            self.reduce_features = nn.Sequential(
                nn.Linear(1280, 64), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(64, N_QUBITS),
            )
            self.bn_reduce  = nn.BatchNorm1d(N_QUBITS)
            self.quantum    = QuantumLayer()
            self.classifier = nn.Linear(N_QUBITS, n_classes)
    
        def forward(self, x):
            x = self.feature_extractor(x)
            x = self.reduce_features(x)
            x = (torch.tanh(x) + 1) * (np.pi / 2)
            x = self.quantum(x)
            x = self.bn_reduce(x)
            return self.classifier(x)
    
        def get_angles(self, x):
            with torch.no_grad():
                x = self.feature_extractor(x)
                x = self.reduce_features(x)
                x = (torch.tanh(x) + 1) * (np.pi / 2)
            return x
    
    model = EfficientNetQuantumClassifier(n_classes=2).to(device_cpu)
    model.load_state_dict(torch.load(BEST_PATH, map_location=device_cpu))
    model.eval()
    print(f"✅ Model loaded from {BEST_PATH}")
    
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 3. REBUILD TEST SET (identical seed/split as training)
    # ══════════════════════════════════════════════════════════════════════════════
    eval_tf = transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    full_dataset = ImageFolder(
        os.path.join(DATA_DIR, "Training/Training"), transform=eval_tf
    )
    class_names = full_dataset.classes
    
    class_indices = defaultdict(list)
    for idx, (_, label) in enumerate(full_dataset.samples):
        class_indices[label].append(idx)
    
    random.seed(SEED)
    train_idx, val_idx, test_idx = [], [], []
    for label in sorted(class_indices.keys()):
        cls_sel = random.sample(class_indices[label], 1000)
        n = len(cls_sel)
        n_train = int(0.70 * n); n_val = int(0.15 * n)
        train_idx += cls_sel[:n_train]
        val_idx   += cls_sel[n_train:n_train + n_val]
        test_idx  += cls_sel[n_train + n_val:]
    
    test_loader = DataLoader(Subset(full_dataset, test_idx),
                             batch_size=32, shuffle=False, num_workers=2)
    print(f"✅ Test set rebuilt: {len(test_idx)} samples")
    
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 4. EXTRACT ANGLE VECTORS + QUANTUM WEIGHTS
    # ══════════════════════════════════════════════════════════════════════════════
    all_angles, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            all_angles.append(model.get_angles(images).numpy())
            all_labels.extend(labels.numpy())
    
    all_angles = np.vstack(all_angles)
    all_labels = np.array(all_labels)
    print(f"✅ Angle vectors: {all_angles.shape}")
    
    trained_weights = None
    for _, param in model.quantum.qlayer.named_parameters():
        trained_weights = param.detach().cpu().numpy()
        break
    
    # Save both for restartability
    np.save(os.path.join(OUT_DIR, "test_angles.npy"),       all_angles)
    np.save(os.path.join(OUT_DIR, "test_labels.npy"),       all_labels)
    np.save(os.path.join(OUT_DIR, "trained_weights.npy"),   trained_weights)
    print(f"✅ Quantum weights: {trained_weights.shape} | saved to {OUT_DIR}")
    
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 5. QISKIT CIRCUIT BUILDER
    # ══════════════════════════════════════════════════════════════════════════════
    def build_vqc(angle_vals, weight_vals):
        """
        4-qubit VQC: AngleEmbedding (RX) + StronglyEntanglingLayers D=1.
        No measurements — EstimatorV2 applies observables directly.
        """
        qc = QiskitCircuit(N_QUBITS)
        for i in range(N_QUBITS):
            qc.rx(float(angle_vals[i]), i)
        w = weight_vals[0]
        for i in range(N_QUBITS):
            qc.rz(float(w[i, 0]), i)
            qc.ry(float(w[i, 1]), i)
            qc.rz(float(w[i, 2]), i)
        for i in range(N_QUBITS):
            qc.cx(i, (i + 1) % N_QUBITS)
        return qc
    
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 6. CONNECT TO IBM + TRANSPILE + LAYOUT-AWARE OBSERVABLES
    # ══════════════════════════════════════════════════════════════════════════════
    print("\n🔌 Connecting to IBM Quantum...")
    secrets = UserSecretsClient()
    service = QiskitRuntimeService(
        channel  = "ibm_quantum_platform",
        token    = secrets.get_secret("IBMTOKEN"),
        instance = secrets.get_secret("IBMCRN")
    )
    
    print("\nAvailable backends:")
    for b in service.backends():
        try:
            status = b.status()
            print(f"  {b.name:25} | queue: {status.pending_jobs:>4} | "
                  f"operational: {status.operational}")
        except Exception:
            print(f"  {b.name:25} | status unavailable")
    
    backend = service.backend("ibm_marrakesh")
    print(f"\n✅ Using backend: {backend.name} ({backend.num_qubits} physical qubits)")
    
    # ── Transpile reference circuit ───────────────────────────────────────────────
    ref_qc   = build_vqc(all_angles[0], trained_weights)
    pm       = generate_preset_pass_manager(backend=backend, optimization_level=1)
    ref_qc_t = pm.run(ref_qc)
    
    n_physical = ref_qc_t.num_qubits
    print(f"\n   Logical qubits  : {N_QUBITS}")
    print(f"   Physical qubits : {n_physical}")
    print(f"   Circuit depth   : {ref_qc_t.depth()}")
    print(f"   Gate count      : {ref_qc_t.size()}")
    
    # ── Gate count breakdown ──────────────────────────────────────────────────────
    gate_counts = dict(ref_qc_t.count_ops())
    print(f"   Native gates    : {gate_counts}")
    
    gate_info = {
        "logical_qubits"   : N_QUBITS,
        "physical_qubits"  : n_physical,
        "circuit_depth"    : ref_qc_t.depth(),
        "total_gates"      : ref_qc_t.size(),
        "gate_breakdown"   : gate_counts,
    }
    with open(os.path.join(OUT_DIR, "gate_counts.json"), "w") as f:
        json.dump(gate_info, f, indent=2)
    print("✅ Gate counts saved")
    
    # ── Circuit diagrams ──────────────────────────────────────────────────────────
    try:
        fig_logical = ref_qc.draw("mpl")
        fig_logical.savefig(os.path.join(OUT_DIR, "circuit_logical.png"),
                            dpi=200, bbox_inches="tight")
        plt.close(fig_logical)
        print("✅ circuit_logical.png")
    except Exception as e:
        print(f"⚠️  Logical circuit draw failed: {e}")
    
    try:
        fig_transpiled = ref_qc_t.draw("mpl", fold=40)
        fig_transpiled.savefig(os.path.join(OUT_DIR, "circuit_transpiled.png"),
                               dpi=200, bbox_inches="tight")
        plt.close(fig_transpiled)
        print("✅ circuit_transpiled.png")
    except Exception as e:
        print(f"⚠️  Transpiled circuit draw failed: {e}")
    
    # ── Physical qubit mapping ────────────────────────────────────────────────────
    layout       = ref_qc_t.layout
    final_layout = layout.final_layout
    
    physical_qubits = []
    for logical_q in range(N_QUBITS):
        phys = final_layout[logical_q]
        if hasattr(phys, 'index'):
            phys = phys.index
        elif hasattr(phys, '_index'):
            phys = phys._index
        else:
            phys = int(str(phys).split('index=')[1].rstrip('>'))
        physical_qubits.append(phys)
    
    print(f"\n   Logical→Physical: {list(zip(range(N_QUBITS), physical_qubits))}")
    
    # ── Build layout-aware observables ───────────────────────────────────────────
    def make_observable(physical_q, n_total):
        pauli_str = ["I"] * n_total
        pauli_str[n_total - 1 - physical_q] = "Z"
        return SparsePauliOp("".join(pauli_str))
    
    observables = [
        make_observable(physical_qubits[i], n_physical)
        for i in range(N_QUBITS)
    ]
    print("✅ Layout-aware observables built")
    for i, obs in enumerate(observables):
        print(f"   Logical Q{i} → Physical Q{physical_qubits[i]} | {obs}")
    
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 6b. NOISE CHARACTERISATION (T1, T2, gate errors for used qubits)
    # ══════════════════════════════════════════════════════════════════════════════
    print("\n📊 Pulling noise characterisation for used physical qubits...")
    noise_data = {}
    try:
        properties = backend.properties()
        for pq in physical_qubits:
            entry = {}
            try:
                entry["T1_us"] = round(properties.t1(pq) * 1e6, 2)
            except Exception:
                entry["T1_us"] = None
            try:
                entry["T2_us"] = round(properties.t2(pq) * 1e6, 2)
            except Exception:
                entry["T2_us"] = None
            try:
                entry["readout_error"] = round(properties.readout_error(pq), 5)
            except Exception:
                entry["readout_error"] = None
            try:
                # single-qubit gate error (sx is the native 1Q gate on Heron)
                entry["sx_error"] = round(properties.gate_error("sx", pq), 5)
            except Exception:
                entry["sx_error"] = None
            noise_data[f"Q{pq}"] = entry
            print(f"   Physical Q{pq}: T1={entry['T1_us']}μs | "
                  f"T2={entry['T2_us']}μs | "
                  f"readout_err={entry['readout_error']} | "
                  f"sx_err={entry['sx_error']}")
    
        # CX / ECR errors between adjacent pairs
        cx_errors = {}
        for i in range(N_QUBITS - 1):
            pair = (physical_qubits[i], physical_qubits[i+1])
            for gate_name in ["cx", "ecr", "cz"]:
                try:
                    err = round(properties.gate_error(gate_name, list(pair)), 5)
                    cx_errors[f"{gate_name}_Q{pair[0]}_Q{pair[1]}"] = err
                    print(f"   {gate_name.upper()} Q{pair[0]}→Q{pair[1]}: {err}")
                    break
                except Exception:
                    continue
    
        noise_data["two_qubit_errors"] = cx_errors
        with open(os.path.join(OUT_DIR, "noise_characterisation.json"), "w") as f:
            json.dump(noise_data, f, indent=2)
        print("✅ noise_characterisation.json saved")
    
        # ── Noise bar chart ───────────────────────────────────────────────────────
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        qubit_labels = [f"Q{pq}" for pq in physical_qubits]
    
        for ax, key, ylabel, title, color in zip(
            axes,
            ["T1_us", "T2_us", "readout_error"],
            ["T1 (μs)", "T2 (μs)", "Readout Error"],
            ["(a) T1 Coherence Times", "(b) T2 Coherence Times",
             "(c) Readout Error Rates"],
            ["#2980b9", "#27ae60", "#e74c3c"]
        ):
            vals = [noise_data.get(f"Q{pq}", {}).get(key) for pq in physical_qubits]
            vals_clean = [v if v is not None else 0 for v in vals]
            bars = ax.bar(qubit_labels, vals_clean, color=color, alpha=0.85,
                          edgecolor="white")
            for bar, val in zip(bars, vals_clean):
                if val > 0:
                    ax.text(bar.get_x() + bar.get_width()/2,
                            bar.get_height() * 1.02,
                            f"{val:.3f}" if key == "readout_error" else f"{val:.1f}",
                            ha="center", va="bottom", fontsize=9)
            ax.set_title(title); ax.set_ylabel(ylabel)
            ax.set_xlabel("Physical Qubit")
    
        fig.suptitle(f"Noise Characterisation — {backend.name} (Physical Qubits Used)",
                     fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "noise_characterisation.png"), dpi=300)
        plt.close()
        print("✅ noise_characterisation.png")
    
    except Exception as e:
        print(f"⚠️  Noise characterisation failed (Heron r2 may not expose properties): {e}")
        print("   Continuing with inference...")
    
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 7. HARDWARE INFERENCE
    # ══════════════════════════════════════════════════════════════════════════════
    def hw_inference_batch(angle_batch, weights, backend, pm, observables, shots):
        estimator = Estimator(backend)
        estimator.options.default_shots = shots
        pubs = []
        for angles in angle_batch:
            qc   = build_vqc(angles, weights)
            qc_t = pm.run(qc)
            for obs in observables:
                pubs.append((qc_t, obs))
        job    = estimator.run(pubs)
        print(f"    Job ID: {job.job_id()}")
        result = job.result()
        evs    = np.array([r.data.evs for r in result])
        return evs.reshape(len(angle_batch), N_QUBITS)
    
    
    print(f"\n🚀 Hardware inference | {len(all_angles)} samples | "
          f"batch={BATCH_HW} | shots={SHOTS}")
    
    hw_evs    = []
    n_total   = len(all_angles)
    n_batches = (n_total + BATCH_HW - 1) // BATCH_HW
    t0        = time.time()
    
    for b in range(n_batches):
        start = b * BATCH_HW
        end   = min(start + BATCH_HW, n_total)
        batch = all_angles[start:end]
        print(f"\n  Batch {b+1}/{n_batches} | samples {start}–{end-1}")
        try:
            evs = hw_inference_batch(
                batch, trained_weights, backend, pm, observables, SHOTS
            )
            hw_evs.append(evs)
            print(f"    ✅ shape={evs.shape} | "
                  f"mean⟨Z⟩={evs.mean(axis=0).round(3)}")
        except Exception as e:
            print(f"    ❌ Failed: {e}")
            hw_evs.append(np.zeros((len(batch), N_QUBITS)))
    
        np.save(os.path.join(OUT_DIR, "hw_evs_partial.npy"), np.vstack(hw_evs))
    
    elapsed = time.time() - t0
    hw_evs  = np.vstack(hw_evs)
    np.save(os.path.join(OUT_DIR, "hw_evs_final.npy"), hw_evs)
    print(f"\n✅ Hardware inference complete in {elapsed/60:.1f} min")
    
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 8. CLASSIFY (BN + Linear head, classical — exact)
    # ══════════════════════════════════════════════════════════════════════════════
    model.eval()
    with torch.no_grad():
        normed   = model.bn_reduce(torch.tensor(hw_evs, dtype=torch.float32))
        logits   = model.classifier(normed)
        hw_probs = torch.softmax(logits, dim=1).numpy()
        hw_preds = np.argmax(hw_probs, axis=1)
    
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 9. SIMULATOR BASELINE
    # ══════════════════════════════════════════════════════════════════════════════
    sim_preds, sim_probs, sim_evs_list = [], [], []
    with torch.no_grad():
        for images, _ in test_loader:
            out  = model(images)
            prob = torch.softmax(out, dim=1).numpy()
            sim_preds.extend(np.argmax(prob, axis=1))
            sim_probs.extend(prob)
            # collect simulator expectation values
            angles_s = model.get_angles(images)
            evs_s    = model.quantum.qlayer(angles_s)
            sim_evs_list.append(evs_s.numpy())
    
    sim_preds = np.array(sim_preds)
    sim_probs = np.array(sim_probs)
    sim_evs   = np.vstack(sim_evs_list)
    
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 10. METRICS
    # ══════════════════════════════════════════════════════════════════════════════
    def compute_metrics(y_true, y_pred, y_probs, label):
        acc  = accuracy_score(y_true, y_pred) * 100
        prec = precision_score(y_true, y_pred, average="macro", zero_division=0) * 100
        rec  = recall_score(y_true,    y_pred, average="macro", zero_division=0) * 100
        f1   = f1_score(y_true,        y_pred, average="macro", zero_division=0) * 100
        try:    auc = roc_auc_score(y_true, y_probs[:, 1]) * 100
        except: auc = float("nan")
        print(f"\n{'='*55}\n  {label}\n{'='*55}")
        print(f"  Accuracy  : {acc:.2f}%")
        print(f"  Precision : {prec:.2f}%")
        print(f"  Recall    : {rec:.2f}%")
        print(f"  F1        : {f1:.2f}%")
        print(f"  AUC       : {auc:.2f}%")
        print(classification_report(y_true, y_pred, target_names=class_names))
        return dict(accuracy=acc, precision=prec, recall=rec, f1=f1, auc=auc)
    
    sim_m = compute_metrics(all_labels, sim_preds, sim_probs,
                            "Simulator (PennyLane default.qubit)")
    hw_m  = compute_metrics(all_labels, hw_preds, hw_probs,
                            f"Real Hardware ({backend.name}, {SHOTS} shots)")
    
    print(f"\n{'='*55}\n  Simulation → Hardware Delta\n{'='*55}")
    for k in sim_m:
        d = hw_m[k] - sim_m[k]
        print(f"  {k:12}: {'+' if d >= 0 else ''}{d:.2f}")
    
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 11. ALL PLOTS
    # ══════════════════════════════════════════════════════════════════════════════
    
    # ── 11a. Confusion matrices side by side ─────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, preds, title in zip(
        axes,
        [sim_preds, hw_preds],
        ["(a) Simulator", f"(b) Real Hardware — {backend.name} ({SHOTS} shots)"]
    ):
        cm     = confusion_matrix(all_labels, preds)
        cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
        sns.heatmap(cm, annot=False, cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names,
                    linewidths=0.5, ax=ax)
        for i in range(2):
            for j in range(2):
                c = "white" if cm_pct[i,j] > 55 else "#1a1a1a"
                ax.text(j+0.5, i+0.38, f"{cm[i,j]}",
                        ha="center", va="center",
                        fontsize=14, fontweight="bold", color=c)
                ax.text(j+0.5, i+0.68, f"({cm_pct[i,j]:.1f}%)",
                        ha="center", va="center", fontsize=9, color=c)
        ax.set_title(f"{title}\nAcc = {np.trace(cm)/cm.sum()*100:.2f}%")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    fig.suptitle("Simulator vs. Real Hardware — Confusion Matrices",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "hw_confusion_matrices.png"), dpi=300)
    plt.close(); print("✅ hw_confusion_matrices.png")
    
    # ── 11b. Metrics bar chart ────────────────────────────────────────────────────
    keys = ["accuracy", "precision", "recall", "f1"]
    x = np.arange(len(keys)); w = 0.35
    fig, ax = plt.subplots(figsize=(10, 5))
    b1 = ax.bar(x - w/2, [sim_m[k] for k in keys], w,
                label="Simulator", color="#2980b9", alpha=0.85, edgecolor="white")
    b2 = ax.bar(x + w/2, [hw_m[k]  for k in keys], w,
                label=f"Real HW ({backend.name})",
                color="#e74c3c", alpha=0.85, edgecolor="white")
    for bar in list(b1) + list(b2):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([k.capitalize() for k in keys])
    ax.set_ylabel("Score (%)"); ax.set_ylim([0, 115])
    ax.set_title("Simulator vs. Real Hardware Performance", fontweight="bold")
    ax.legend(); ax.axhline(y=100, color="k", lw=0.5, ls="--", alpha=0.2)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "hw_metrics_comparison.png"), dpi=300)
    plt.close(); print("✅ hw_metrics_comparison.png")
    
    # ── 11c. Expectation value distributions per qubit ───────────────────────────
    fig, axes = plt.subplots(1, N_QUBITS, figsize=(14, 4))
    for i, ax in enumerate(axes):
        ax.hist(sim_evs[:, i], bins=25, alpha=0.6,
                color="#2980b9", label="Simulator", density=True)
        ax.hist(hw_evs[:, i],  bins=25, alpha=0.6,
                color="#e74c3c", label="Real HW",   density=True)
        ax.axvline(x=sim_evs[:, i].mean(), color="#2980b9",
                   lw=1.5, ls="--", alpha=0.8,
                   label=f"Sim μ={sim_evs[:,i].mean():.2f}")
        ax.axvline(x=hw_evs[:, i].mean(),  color="#e74c3c",
                   lw=1.5, ls="--", alpha=0.8,
                   label=f"HW μ={hw_evs[:,i].mean():.2f}")
        ax.set_title(f"Qubit {i} — ⟨Z⟩")
        ax.set_xlabel("Expectation value"); ax.set_ylabel("Density")
        ax.legend(fontsize=7)
    fig.suptitle("⟨Z⟩ Distributions per Qubit: Simulator vs. Real Hardware",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "hw_expectation_distributions.png"), dpi=300)
    plt.close(); print("✅ hw_expectation_distributions.png")
    
    # ── 11d. Noise-induced expectation value shift (scatter) ─────────────────────
    fig, axes = plt.subplots(1, N_QUBITS, figsize=(14, 4))
    for i, ax in enumerate(axes):
        ax.scatter(sim_evs[:, i], hw_evs[:, i],
                   alpha=0.3, s=12, color="#8e44ad")
        lims = [
            min(sim_evs[:, i].min(), hw_evs[:, i].min()) - 0.05,
            max(sim_evs[:, i].max(), hw_evs[:, i].max()) + 0.05
        ]
        ax.plot(lims, lims, "k--", lw=1, alpha=0.5, label="Ideal (no noise)")
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel("Simulator ⟨Z⟩")
        ax.set_ylabel("Hardware ⟨Z⟩")
        ax.set_title(f"Qubit {i}")
        ax.legend(fontsize=7)
        corr = np.corrcoef(sim_evs[:, i], hw_evs[:, i])[0, 1]
        ax.text(0.05, 0.92, f"r={corr:.3f}",
                transform=ax.transAxes, fontsize=9,
                bbox=dict(boxstyle="round", fc="white", alpha=0.7))
    fig.suptitle("Simulator vs. Hardware ⟨Z⟩ Correlation per Qubit",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "hw_expectation_scatter.png"), dpi=300)
    plt.close(); print("✅ hw_expectation_scatter.png")
    
    # ── 11e. Gate count summary bar chart ────────────────────────────────────────
    if gate_counts:
        fig, ax = plt.subplots(figsize=(8, 4))
        gates = list(gate_counts.keys())
        counts = list(gate_counts.values())
        bars = ax.bar(gates, counts, color="#2c3e50", alpha=0.8, edgecolor="white")
        for bar, val in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                    str(val), ha="center", va="bottom", fontsize=10)
        ax.set_xlabel("Gate Type"); ax.set_ylabel("Count")
        ax.set_title(f"Transpiled Circuit Gate Breakdown — {backend.name}\n"
                     f"Depth={ref_qc_t.depth()} | Total gates={ref_qc_t.size()}",
                     fontweight="bold")
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "hw_gate_counts.png"), dpi=300)
        plt.close(); print("✅ hw_gate_counts.png")
    
    
    # ══════════════════════════════════════════════════════════════════════════════
    # 12. SAVE ALL RESULTS
    # ══════════════════════════════════════════════════════════════════════════════
    results = {
        "backend"          : backend.name,
        "shots"            : SHOTS,
        "n_test_samples"   : int(len(all_labels)),
        "circuit_info"     : gate_info,
        "physical_qubit_mapping": list(zip(range(N_QUBITS), physical_qubits)),
        "simulator_metrics": {k: round(v, 4) for k, v in sim_m.items()},
        "hardware_metrics" : {k: round(v, 4) for k, v in hw_m.items()},
        "delta"            : {k: round(hw_m[k] - sim_m[k], 4) for k in sim_m},
        "elapsed_min"      : round(elapsed / 60, 2),
        "expectation_value_stats": {
            f"Q{i}": {
                "sim_mean" : round(float(sim_evs[:, i].mean()), 4),
                "sim_std"  : round(float(sim_evs[:, i].std()),  4),
                "hw_mean"  : round(float(hw_evs[:, i].mean()),  4),
                "hw_std"   : round(float(hw_evs[:, i].std()),   4),
                "shift"    : round(float(hw_evs[:, i].mean() - sim_evs[:, i].mean()), 4),
            }
            for i in range(N_QUBITS)
        }
    }
    
    with open(os.path.join(OUT_DIR, "hardware_results.json"), "w") as f:
        json.dump(results, f, indent=2)
    
    print(f"\n{'='*55}")
    print("  HARDWARE EVALUATION COMPLETE")
    print(f"{'='*55}")
    print(f"  Simulator accuracy : {sim_m['accuracy']:.2f}%")
    print(f"  Hardware  accuracy : {hw_m['accuracy']:.2f}%")
    print(f"  Accuracy drop      : {hw_m['accuracy'] - sim_m['accuracy']:.2f}%")
    print(f"  Elapsed            : {elapsed/60:.1f} min")
    print(f"\n  Output files in: {OUT_DIR}/")
    print(f"    hardware_results.json")
    print(f"    gate_counts.json")
    print(f"    noise_characterisation.json  (if backend.properties() available)")
    print(f"    circuit_logical.png")
    print(f"    circuit_transpiled.png")
    print(f"    hw_confusion_matrices.png")
    print(f"    hw_metrics_comparison.png")
    print(f"    hw_expectation_distributions.png")
    print(f"    hw_expectation_scatter.png")
    print(f"    hw_gate_counts.png")
    print(f"    noise_characterisation.png   (if backend.properties() available)")

Sole Hardware Run ()
: Create IBM account at https://quantum.cloud.ibm.com/
: Extract "API KEY" and "CRN" (44 lenght string) with names "IBMTOKEN" and "IBMCRN" respectively. 
: IBM Open plan (10 min free / month)
: Add as "Secrets" in kaggle (Click Add-ons -> Secrets)
: You are good to go with the below code. 

For any queries or issues, feel free to ping at "tarun232770@cuh.ac.in"

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=secrets.get_secret("IBMTOKEN3"),    # reads from Kaggle Secrets
    instance=secrets.get_secret("IBMCRN3")   # reads from Kaggle Secrets
)

# Test it works:
print(service.backends())  # should print a list of quantum computers

# Check queue lengths — pick the one with fewest pending jobs
for b in service.backends():
    status = b.status()
    print(f"{b.name:20} | jobs in queue: {status.pending_jobs:>4} | operational: {status.operational}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# HARDWARE EVALUATION v4 — EfficientNet-B0 + QNN on IBM Quantum
# Features:
#   ✅ Best qubit chain selection (noise-aware)
#   ✅ Built-in job polling with timeout (no silent hangs)
#   ✅ Per-batch retry (up to MAX_RETRIES attempts)
#   ✅ Resume from partial save (kernel-restart safe)
#   ✅ Auto fallback backend if quota exhausted
#   ✅ Circuit diagrams, gate counts, noise characterisation, all plots
# ══════════════════════════════════════════════════════════════════════════════

import os, json, time, random, warnings
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.models import efficientnet_b0
from torch.utils.data import DataLoader, Subset
from collections import defaultdict

import pennylane as qml

from qiskit import QuantumCircuit as QiskitCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler import Layout
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
from qiskit.quantum_info import SparsePauliOp

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, classification_report
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from kaggle_secrets import UserSecretsClient

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("✅ All imports OK")


# ══════════════════════════════════════════════════════════════════════════════
# 1. CONFIG
# ══════════════════════════════════════════════════════════════════════════════
N_QUBITS      = 4
Q_DEPTH       = 1
SHOTS         = 1024
BATCH_HW      = 10

# ── Retry / polling settings ──────────────────────────────────────────────────
MAX_RETRIES   = 3          # how many times to retry a failed batch
JOB_TIMEOUT_S = 300        # seconds before cancelling a stuck job
POLL_INTERVAL = 15         # seconds between status checks

# ── Preferred backend + ordered fallbacks ────────────────────────────────────
BACKEND_PRIORITY = ["ibm_fez", "ibm_marrakesh", "ibm_kingston"]

BEST_PATH  = "/kaggle/working/best_model_quantum.pth"
DATA_DIR   = "/kaggle/input/datasets/tarunshr/forest-fire/"
OUT_DIR    = "/kaggle/working/hardware_eval"
os.makedirs(OUT_DIR, exist_ok=True)

PARTIAL_PATH = os.path.join(OUT_DIR, "hw_evs_partial.npy")
DONE_PATH    = os.path.join(OUT_DIR, "batches_done.json")   # tracks completed batches

device_cpu = torch.device("cpu")


# ══════════════════════════════════════════════════════════════════════════════
# 2. MODEL
# ══════════════════════════════════════════════════════════════════════════════
base_model = efficientnet_b0(weights="DEFAULT")
base_model.classifier = nn.Identity()

dev_pl = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev_pl, interface="torch")
def quantum_circuit(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qml.qnn.TorchLayer(
            quantum_circuit, {"weights": (Q_DEPTH, N_QUBITS, 3)}
        )
    def forward(self, x):
        return self.qlayer(x)

class EfficientNetQuantumClassifier(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.feature_extractor = base_model
        self.reduce_features = nn.Sequential(
            nn.Linear(1280, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, N_QUBITS),
        )
        self.bn_reduce  = nn.BatchNorm1d(N_QUBITS)
        self.quantum    = QuantumLayer()
        self.classifier = nn.Linear(N_QUBITS, n_classes)

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.reduce_features(x)
        x = (torch.tanh(x) + 1) * (np.pi / 2)
        x = self.quantum(x)
        x = self.bn_reduce(x)
        return self.classifier(x)

    def get_angles(self, x):
        with torch.no_grad():
            x = self.feature_extractor(x)
            x = self.reduce_features(x)
            x = (torch.tanh(x) + 1) * (np.pi / 2)
        return x

model = EfficientNetQuantumClassifier(n_classes=2).to(device_cpu)
model.load_state_dict(torch.load(BEST_PATH, map_location=device_cpu))
model.eval()
print(f"✅ Model loaded from {BEST_PATH}")


# ══════════════════════════════════════════════════════════════════════════════
# 3. TEST SET
# ══════════════════════════════════════════════════════════════════════════════
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
full_dataset = ImageFolder(
    os.path.join(DATA_DIR, "Training/Training"), transform=eval_tf
)
class_names = full_dataset.classes

class_indices = defaultdict(list)
for idx, (_, label) in enumerate(full_dataset.samples):
    class_indices[label].append(idx)

random.seed(SEED)
train_idx, val_idx, test_idx = [], [], []
for label in sorted(class_indices.keys()):
    cls_sel = random.sample(class_indices[label], 1000)
    n = len(cls_sel)
    n_train = int(0.70 * n); n_val = int(0.15 * n)
    train_idx += cls_sel[:n_train]
    val_idx   += cls_sel[n_train:n_train + n_val]
    test_idx  += cls_sel[n_train + n_val:]

test_loader = DataLoader(Subset(full_dataset, test_idx),
                         batch_size=32, shuffle=False, num_workers=2)
print(f"✅ Test set rebuilt: {len(test_idx)} samples")


# ══════════════════════════════════════════════════════════════════════════════
# 4. ANGLES + WEIGHTS  (load from cache if available)
# ══════════════════════════════════════════════════════════════════════════════
angles_path  = os.path.join(OUT_DIR, "test_angles.npy")
labels_path  = os.path.join(OUT_DIR, "test_labels.npy")
weights_path = os.path.join(OUT_DIR, "trained_weights.npy")

if all(os.path.exists(p) for p in [angles_path, labels_path, weights_path]):
    all_angles      = np.load(angles_path)
    all_labels      = np.load(labels_path)
    trained_weights = np.load(weights_path)
    print(f"✅ Loaded cached angles/labels/weights from {OUT_DIR}")
else:
    all_angles, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            all_angles.append(model.get_angles(images).numpy())
            all_labels.extend(labels.numpy())
    all_angles = np.vstack(all_angles)
    all_labels = np.array(all_labels)

    trained_weights = None
    for _, param in model.quantum.qlayer.named_parameters():
        trained_weights = param.detach().cpu().numpy()
        break

    np.save(angles_path,  all_angles)
    np.save(labels_path,  all_labels)
    np.save(weights_path, trained_weights)
    print(f"✅ Angle vectors: {all_angles.shape} | saved")

print(f"   Weights shape: {trained_weights.shape}")


# ══════════════════════════════════════════════════════════════════════════════
# 5. QISKIT CIRCUIT BUILDER
# ══════════════════════════════════════════════════════════════════════════════
def build_vqc(angle_vals, weight_vals):
    """4-qubit VQC: AngleEmbedding (RX) + StronglyEntanglingLayers D=1."""
    qc = QiskitCircuit(N_QUBITS)
    for i in range(N_QUBITS):
        qc.rx(float(angle_vals[i]), i)
    w = weight_vals[0]
    for i in range(N_QUBITS):
        qc.rz(float(w[i, 0]), i)
        qc.ry(float(w[i, 1]), i)
        qc.rz(float(w[i, 2]), i)
    for i in range(N_QUBITS):
        qc.cx(i, (i + 1) % N_QUBITS)
    return qc


# ══════════════════════════════════════════════════════════════════════════════
# 6. IBM CONNECTION + BACKEND SELECTION
# ══════════════════════════════════════════════════════════════════════════════
print("\n🔌 Connecting to IBM Quantum...")
secrets = UserSecretsClient()
service = QiskitRuntimeService(
    channel  = "ibm_quantum_platform",
    token    = secrets.get_secret("IBMTOKEN4"),
    instance = secrets.get_secret("IBMCRN4")
)

# ── Check quota ───────────────────────────────────────────────────────────────
usage = service.usage()
print(f"\n📊 Quota: {usage['usage_consumed_seconds']}s used / "
      f"{usage['usage_limit_seconds']}s | "
      f"remaining: {usage['usage_remaining_seconds']}s | "
      f"reset: {usage['usage_period']['end_time']}")
if usage['usage_limit_reached']:
    raise SystemExit("❌ Quota exhausted — wait for monthly reset before running hardware jobs.")

# ── List backends + pick best available ──────────────────────────────────────
print("\nAvailable backends:")
available = {}
for b in service.backends():
    try:
        status = b.status()
        print(f"  {b.name:25} | queue: {status.pending_jobs:>4} | "
              f"operational: {status.operational}")
        if status.operational:
            available[b.name] = status.pending_jobs
    except Exception:
        print(f"  {b.name:25} | status unavailable")

backend_name = None
for preferred in BACKEND_PRIORITY:
    if preferred in available:
        backend_name = preferred
        break

if backend_name is None:
    raise RuntimeError("No operational backend found in priority list.")

backend = service.backend(backend_name)
print(f"\n✅ Using backend: {backend.name} ({backend.num_qubits} physical qubits)")


# ══════════════════════════════════════════════════════════════════════════════
# 7. BEST QUBIT CHAIN SELECTION  (noise-aware)
# ══════════════════════════════════════════════════════════════════════════════
coupling_map = backend.coupling_map
edges = set(coupling_map.get_edges())

def find_chains(edges, n=4):
    adj = defaultdict(set)
    for a, b in edges:
        adj[a].add(b); adj[b].add(a)
    chains = []
    total_qubits = max(max(a, b) for a, b in edges) + 1
    for start in range(total_qubits):
        def dfs(path):
            if len(path) == n:
                chains.append(list(path)); return
            for nb in adj[path[-1]]:
                if nb not in path:
                    dfs(path + [nb])
        dfs([start])
    return chains

chains = find_chains(edges)
print(f"\nTotal valid {N_QUBITS}-qubit chains: {len(chains)}")

properties = backend.properties()
best_chains = []
for chain in chains:
    try:
        readout_errs = [properties.readout_error(q) for q in chain]
        sx_errs      = [properties.gate_error("sx", q) for q in chain]
        cz_errs = []
        for i in range(len(chain) - 1):
            for gate_name in ["cz", "ecr", "cx"]:
                try:
                    cz_errs.append(properties.gate_error(gate_name, [chain[i], chain[i+1]]))
                    break
                except Exception:
                    continue
            else:
                cz_errs.append(0.01)  # penalty if gate data unavailable
        score = sum(readout_errs) + sum(sx_errs) + sum(cz_errs)
        best_chains.append((score, chain, readout_errs, cz_errs))
    except Exception:
        continue

best_chains.sort()
print(f"\nTop 10 lowest-noise {N_QUBITS}-qubit chains:")
print(f"{'Rank':<5} {'Chain':<22} {'Readout Errs':<42} {'CZ Errs':<35} {'Score'}")
print("-" * 115)
for rank, (score, chain, ro, cz) in enumerate(best_chains[:10], 1):
    ro_str = [f"{e:.4f}" for e in ro]
    cz_str = [f"{e:.4f}" for e in cz]
    print(f"{rank:<5} {str(chain):<22} {str(ro_str):<42} {str(cz_str):<35} {score:.5f}")

CHOSEN_PHYSICAL_QUBITS = best_chains[0][1]
print(f"\n✅ Chosen physical qubits: {CHOSEN_PHYSICAL_QUBITS}")


# ══════════════════════════════════════════════════════════════════════════════
# 8. TRANSPILE + OBSERVABLES
# ══════════════════════════════════════════════════════════════════════════════
ref_qc = build_vqc(all_angles[0], trained_weights)

initial_layout = Layout({
    ref_qc.qubits[i]: CHOSEN_PHYSICAL_QUBITS[i]
    for i in range(N_QUBITS)
})
pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=1,
    initial_layout=initial_layout
)
ref_qc_t     = pm.run(ref_qc)
n_physical   = ref_qc_t.num_qubits
physical_qubits = CHOSEN_PHYSICAL_QUBITS

print(f"\n   Logical qubits  : {N_QUBITS}")
print(f"   Physical qubits : {physical_qubits}")
print(f"   Circuit depth   : {ref_qc_t.depth()}")
print(f"   Gate count      : {ref_qc_t.size()}")

gate_counts = dict(ref_qc_t.count_ops())
print(f"   Native gates    : {gate_counts}")

gate_info = {
    "backend"          : backend.name,
    "logical_qubits"   : N_QUBITS,
    "physical_qubits"  : n_physical,
    "chosen_qubits"    : physical_qubits,
    "circuit_depth"    : ref_qc_t.depth(),
    "total_gates"      : ref_qc_t.size(),
    "gate_breakdown"   : gate_counts,
}
with open(os.path.join(OUT_DIR, "gate_counts.json"), "w") as f:
    json.dump(gate_info, f, indent=2)
print("✅ Gate counts saved")

# ── Circuit diagrams ──────────────────────────────────────────────────────────
for qc, name in [(ref_qc, "circuit_logical"), (ref_qc_t, "circuit_transpiled")]:
    try:
        kw = {"fold": 40} if name == "circuit_transpiled" else {}
        fig = qc.draw("mpl", **kw)
        fig.savefig(os.path.join(OUT_DIR, f"{name}.png"), dpi=200, bbox_inches="tight")
        plt.close(fig)
        print(f"✅ {name}.png")
    except Exception as e:
        print(f"⚠️  {name} draw failed: {e}")

# ── Layout-aware observables ──────────────────────────────────────────────────
def make_observable(physical_q, n_total):
    pauli_str = ["I"] * n_total
    pauli_str[n_total - 1 - physical_q] = "Z"
    return SparsePauliOp("".join(pauli_str))

observables = [make_observable(physical_qubits[i], n_physical) for i in range(N_QUBITS)]
print("✅ Layout-aware observables built")
for i, obs in enumerate(observables):
    print(f"   Logical Q{i} → Physical Q{physical_qubits[i]}")


# ══════════════════════════════════════════════════════════════════════════════
# 9. NOISE CHARACTERISATION
# ══════════════════════════════════════════════════════════════════════════════
print("\n📊 Pulling noise characterisation...")
noise_data = {}
try:
    for pq in physical_qubits:
        entry = {}
        for attr, method, scale in [
            ("T1_us",        lambda q: properties.t1(q) * 1e6, None),
            ("T2_us",        lambda q: properties.t2(q) * 1e6, None),
            ("readout_error",lambda q: properties.readout_error(q), None),
            ("sx_error",     lambda q: properties.gate_error("sx", q), None),
        ]:
            try:    entry[attr] = round(method(pq), 5)
            except: entry[attr] = None
        noise_data[f"Q{pq}"] = entry
        print(f"   Q{pq}: T1={entry['T1_us']}μs | T2={entry['T2_us']}μs | "
              f"readout={entry['readout_error']} | sx={entry['sx_error']}")

    cx_errors = {}
    for i in range(N_QUBITS - 1):
        pair = (physical_qubits[i], physical_qubits[i+1])
        for gate_name in ["cz", "ecr", "cx"]:
            try:
                err = round(properties.gate_error(gate_name, list(pair)), 5)
                cx_errors[f"{gate_name}_Q{pair[0]}_Q{pair[1]}"] = err
                print(f"   {gate_name.upper()} Q{pair[0]}→Q{pair[1]}: {err}")
                break
            except Exception:
                continue

    noise_data["two_qubit_errors"] = cx_errors
    with open(os.path.join(OUT_DIR, "noise_characterisation.json"), "w") as f:
        json.dump(noise_data, f, indent=2)
    print("✅ noise_characterisation.json saved")

    # ── Noise bar chart ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    qubit_labels = [f"Q{pq}" for pq in physical_qubits]
    for ax, key, ylabel, title, color in zip(
        axes,
        ["T1_us", "T2_us", "readout_error"],
        ["T1 (μs)", "T2 (μs)", "Readout Error"],
        ["(a) T1 Coherence", "(b) T2 Coherence", "(c) Readout Error"],
        ["#2980b9", "#27ae60", "#e74c3c"]
    ):
        vals = [noise_data.get(f"Q{pq}", {}).get(key) or 0 for pq in physical_qubits]
        bars = ax.bar(qubit_labels, vals, color=color, alpha=0.85, edgecolor="white")
        for bar, val in zip(bars, vals):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                        f"{val:.3f}" if key == "readout_error" else f"{val:.1f}",
                        ha="center", va="bottom", fontsize=9)
        ax.set_title(title); ax.set_ylabel(ylabel); ax.set_xlabel("Physical Qubit")
    fig.suptitle(f"Noise Characterisation — {backend.name}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "noise_characterisation.png"), dpi=300)
    plt.close()
    print("✅ noise_characterisation.png")

except Exception as e:
    print(f"⚠️  Noise characterisation failed: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# 10. PRE-FLIGHT CHECK
# ══════════════════════════════════════════════════════════════════════════════
print("\n🔍 Pre-flight sanity check (|+⟩ → expect ⟨Z⟩ ≈ 0)...")
test_qc = QiskitCircuit(N_QUBITS)
for i in range(N_QUBITS):
    test_qc.h(i)
test_qc_t  = pm.run(test_qc)
estimator  = Estimator(backend)
estimator.options.default_shots = SHOTS
pubs       = [(test_qc_t, obs) for obs in observables]
result     = estimator.run(pubs).result()
evs        = np.array([r.data.evs for r in result])

all_ok = True
for i in range(N_QUBITS):
    ok = abs(evs[i]) < 0.3
    print(f"  Q{i}: {evs[i]:+.4f}  {'✅' if ok else '❌ observable mismatch!'}")
    if not ok: all_ok = False

if not all_ok:
    raise RuntimeError("Observable check failed — fix layout before running inference.")
print("✅ Pre-flight passed — starting inference\n")


# ══════════════════════════════════════════════════════════════════════════════
# 11. HELPERS: POLLING + PER-BATCH INFERENCE WITH RETRY
# ══════════════════════════════════════════════════════════════════════════════

def poll_job(job, timeout_s=JOB_TIMEOUT_S, poll_interval=POLL_INTERVAL):
    """
    Block until job finishes, times out, or errors.
    Returns (result, True) on success, (None, False) on failure/timeout.
    """
    t0 = time.time()
    while True:
        elapsed    = time.time() - t0
        status_str = str(job.status()).upper()

        if any(s in status_str for s in ["DONE", "COMPLETED"]):
            return job.result(), True

        if any(s in status_str for s in ["ERROR", "FAILED", "CANCELLED"]):
            print(f"    ❌ Job ended with status: {status_str}")
            return None, False

        if elapsed > timeout_s:
            print(f"    ⏰ Timeout after {timeout_s}s — cancelling job")
            try: job.cancel()
            except Exception: pass
            return None, False

        print(f"    [{elapsed:>5.0f}s] {status_str} ...")
        time.sleep(poll_interval)


def run_batch_with_retry(angle_batch, weights, backend, pm, observables,
                         shots, max_retries=MAX_RETRIES):
    """
    Submit a batch to hardware, poll for result, retry up to max_retries times.
    Returns np.array of shape (batch_size, N_QUBITS), or zeros on total failure.
    """
    for attempt in range(1, max_retries + 1):
        try:
            estimator = Estimator(backend)
            estimator.options.default_shots = shots
            pubs = []
            for angles in angle_batch:
                qc   = build_vqc(angles, weights)
                qc_t = pm.run(qc)
                for obs in observables:
                    pubs.append((qc_t, obs))

            job = estimator.run(pubs)
            print(f"    Job ID: {job.job_id()} (attempt {attempt}/{max_retries})")

            result, success = poll_job(job)
            if success:
                evs = np.array([r.data.evs for r in result])
                return evs.reshape(len(angle_batch), N_QUBITS), True

            print(f"    ⚠️  Attempt {attempt} failed — {'retrying' if attempt < max_retries else 'giving up'}")

        except Exception as e:
            print(f"    ❌ Exception on attempt {attempt}: {e}")
            if attempt == max_retries:
                break
            time.sleep(5)

    return np.zeros((len(angle_batch), N_QUBITS)), False


# ══════════════════════════════════════════════════════════════════════════════
# 12. HARDWARE INFERENCE  (resume-aware)
# ══════════════════════════════════════════════════════════════════════════════
n_total   = len(all_angles)
n_batches = (n_total + BATCH_HW - 1) // BATCH_HW

# ── Load partial results + completed-batch tracker ───────────────────────────
if os.path.exists(PARTIAL_PATH):
    hw_evs = np.load(PARTIAL_PATH)
    print(f"♻️  Loaded partial EVs from disk: {hw_evs.shape}")
else:
    hw_evs = np.zeros((n_total, N_QUBITS))

if os.path.exists(DONE_PATH):
    with open(DONE_PATH) as f:
        batches_done = set(json.load(f))
    print(f"♻️  Already completed batches: {sorted(batches_done)}")
else:
    batches_done = set()

# ── Identify which batches still need running ─────────────────────────────────
# A batch is "done" if it's in batches_done AND its rows are non-zero
def batch_is_done(b):
    if b in batches_done:
        start = b * BATCH_HW
        end   = min(start + BATCH_HW, n_total)
        return not (hw_evs[start:end] == 0).all(axis=1).any()
    return False

pending_batches = [b for b in range(n_batches) if not batch_is_done(b)]
print(f"\n🚀 Hardware inference | {n_total} samples | batch={BATCH_HW} | shots={SHOTS}")
print(f"   Pending batches : {len(pending_batches)} / {n_batches}")
print(f"   Backend         : {backend.name}\n")

t0 = time.time()
failed_batches = []

for b in pending_batches:
    start = b * BATCH_HW
    end   = min(start + BATCH_HW, n_total)
    batch = all_angles[start:end]

    print(f"\n  Batch {b+1}/{n_batches} | samples {start}–{end-1}")
    evs, success = run_batch_with_retry(
        batch, trained_weights, backend, pm, observables, SHOTS
    )

    hw_evs[start:end] = evs

    if success:
        batches_done.add(b)
        print(f"    ✅ shape={evs.shape} | mean⟨Z⟩={evs.mean(axis=0).round(3)}")
    else:
        failed_batches.append(b)
        print(f"    ⚠️  Batch {b} stored as zeros (will show in final report)")

    # Persist after every batch
    np.save(PARTIAL_PATH, hw_evs)
    with open(DONE_PATH, "w") as f:
        json.dump(sorted(batches_done), f)

elapsed = time.time() - t0

# ── Final save ────────────────────────────────────────────────────────────────
np.save(os.path.join(OUT_DIR, "hw_evs_final.npy"), hw_evs)
print(f"\n✅ Hardware inference complete in {elapsed/60:.1f} min")
if failed_batches:
    print(f"   ⚠️  Permanently failed batches (stored as zeros): {failed_batches}")


# ══════════════════════════════════════════════════════════════════════════════
# 13. CLASSIFY (BN + Linear head — classical, exact)
# ══════════════════════════════════════════════════════════════════════════════
model.eval()
with torch.no_grad():
    normed   = model.bn_reduce(torch.tensor(hw_evs, dtype=torch.float32))
    logits   = model.classifier(normed)
    hw_probs = torch.softmax(logits, dim=1).numpy()
    hw_preds = np.argmax(hw_probs, axis=1)


# ══════════════════════════════════════════════════════════════════════════════
# 14. SIMULATOR BASELINE
# ══════════════════════════════════════════════════════════════════════════════
sim_preds, sim_probs, sim_evs_list = [], [], []
with torch.no_grad():
    for images, _ in test_loader:
        out  = model(images)
        prob = torch.softmax(out, dim=1).numpy()
        sim_preds.extend(np.argmax(prob, axis=1))
        sim_probs.extend(prob)
        angles_s = model.get_angles(images)
        evs_s    = model.quantum.qlayer(angles_s)
        sim_evs_list.append(evs_s.numpy())

sim_preds = np.array(sim_preds)
sim_probs = np.array(sim_probs)
sim_evs   = np.vstack(sim_evs_list)


# ══════════════════════════════════════════════════════════════════════════════
# 15. METRICS
# ══════════════════════════════════════════════════════════════════════════════
def compute_metrics(y_true, y_pred, y_probs, label):
    acc  = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0) * 100
    rec  = recall_score(y_true,    y_pred, average="macro", zero_division=0) * 100
    f1   = f1_score(y_true,        y_pred, average="macro", zero_division=0) * 100
    try:    auc = roc_auc_score(y_true, y_probs[:, 1]) * 100
    except: auc = float("nan")
    print(f"\n{'='*55}\n  {label}\n{'='*55}")
    print(f"  Accuracy  : {acc:.2f}%")
    print(f"  Precision : {prec:.2f}%")
    print(f"  Recall    : {rec:.2f}%")
    print(f"  F1        : {f1:.2f}%")
    print(f"  AUC       : {auc:.2f}%")
    print(classification_report(y_true, y_pred, target_names=class_names))
    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1, auc=auc)

sim_m = compute_metrics(all_labels, sim_preds, sim_probs,
                        "Simulator (PennyLane default.qubit)")
hw_m  = compute_metrics(all_labels, hw_preds, hw_probs,
                        f"Real Hardware ({backend.name}, {SHOTS} shots)")

print(f"\n{'='*55}\n  Simulation → Hardware Delta\n{'='*55}")
for k in sim_m:
    d = hw_m[k] - sim_m[k]
    print(f"  {k:12}: {'+' if d >= 0 else ''}{d:.2f}")


# ══════════════════════════════════════════════════════════════════════════════
# 16. PLOTS
# ══════════════════════════════════════════════════════════════════════════════

# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, preds, title in zip(
    axes,
    [sim_preds, hw_preds],
    ["(a) Simulator", f"(b) Real Hardware — {backend.name} ({SHOTS} shots)"]
):
    cm     = confusion_matrix(all_labels, preds)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm, annot=False, cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    for i in range(2):
        for j in range(2):
            c = "white" if cm_pct[i,j] > 55 else "#1a1a1a"
            ax.text(j+0.5, i+0.38, f"{cm[i,j]}",
                    ha="center", va="center", fontsize=14, fontweight="bold", color=c)
            ax.text(j+0.5, i+0.68, f"({cm_pct[i,j]:.1f}%)",
                    ha="center", va="center", fontsize=9, color=c)
    ax.set_title(f"{title}\nAcc={np.trace(cm)/cm.sum()*100:.2f}%")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
fig.suptitle("Simulator vs. Real Hardware — Confusion Matrices", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "hw_confusion_matrices.png"), dpi=300)
plt.close(); print("✅ hw_confusion_matrices.png")

# ── Metrics bar chart ─────────────────────────────────────────────────────────
keys = ["accuracy", "precision", "recall", "f1"]
x = np.arange(len(keys)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, [sim_m[k] for k in keys], w,
            label="Simulator", color="#2980b9", alpha=0.85, edgecolor="white")
b2 = ax.bar(x + w/2, [hw_m[k]  for k in keys], w,
            label=f"Real HW ({backend.name})", color="#e74c3c", alpha=0.85, edgecolor="white")
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([k.capitalize() for k in keys])
ax.set_ylabel("Score (%)"); ax.set_ylim([0, 115])
ax.set_title("Simulator vs. Real Hardware Performance", fontweight="bold")
ax.legend(); ax.axhline(y=100, color="k", lw=0.5, ls="--", alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "hw_metrics_comparison.png"), dpi=300)
plt.close(); print("✅ hw_metrics_comparison.png")

# ── ⟨Z⟩ distributions ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, N_QUBITS, figsize=(14, 4))
for i, ax in enumerate(axes):
    ax.hist(sim_evs[:, i], bins=25, alpha=0.6, color="#2980b9", label="Simulator", density=True)
    ax.hist(hw_evs[:,  i], bins=25, alpha=0.6, color="#e74c3c", label="Real HW",   density=True)
    ax.axvline(x=sim_evs[:, i].mean(), color="#2980b9", lw=1.5, ls="--",
               label=f"Sim μ={sim_evs[:,i].mean():.2f}")
    ax.axvline(x=hw_evs[:,  i].mean(), color="#e74c3c", lw=1.5, ls="--",
               label=f"HW  μ={hw_evs[:,i].mean():.2f}")
    ax.set_title(f"Qubit {i} ⟨Z⟩")
    ax.set_xlabel("Expectation value"); ax.set_ylabel("Density"); ax.legend(fontsize=7)
fig.suptitle("⟨Z⟩ Distributions: Simulator vs. Real Hardware", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "hw_expectation_distributions.png"), dpi=300)
plt.close(); print("✅ hw_expectation_distributions.png")

# ── Scatter correlation ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, N_QUBITS, figsize=(14, 4))
for i, ax in enumerate(axes):
    ax.scatter(sim_evs[:, i], hw_evs[:, i], alpha=0.3, s=12, color="#8e44ad")
    lims = [min(sim_evs[:,i].min(), hw_evs[:,i].min()) - 0.05,
            max(sim_evs[:,i].max(), hw_evs[:,i].max()) + 0.05]
    ax.plot(lims, lims, "k--", lw=1, alpha=0.5, label="Ideal")
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("Simulator ⟨Z⟩"); ax.set_ylabel("Hardware ⟨Z⟩")
    ax.set_title(f"Qubit {i}"); ax.legend(fontsize=7)
    corr = np.corrcoef(sim_evs[:, i], hw_evs[:, i])[0, 1]
    ax.text(0.05, 0.92, f"r={corr:.3f}", transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle="round", fc="white", alpha=0.7))
fig.suptitle("Simulator vs. Hardware ⟨Z⟩ Correlation", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "hw_expectation_scatter.png"), dpi=300)
plt.close(); print("✅ hw_expectation_scatter.png")

# ── Gate counts bar chart ─────────────────────────────────────────────────────
if gate_counts:
    fig, ax = plt.subplots(figsize=(8, 4))
    gates = list(gate_counts.keys()); counts = list(gate_counts.values())
    bars = ax.bar(gates, counts, color="#2c3e50", alpha=0.8, edgecolor="white")
    for bar, val in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                str(val), ha="center", va="bottom", fontsize=10)
    ax.set_xlabel("Gate Type"); ax.set_ylabel("Count")
    ax.set_title(f"Transpiled Circuit Gate Breakdown — {backend.name}\n"
                 f"Depth={ref_qc_t.depth()} | Total gates={ref_qc_t.size()}",
                 fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "hw_gate_counts.png"), dpi=300)
    plt.close(); print("✅ hw_gate_counts.png")


# ══════════════════════════════════════════════════════════════════════════════
# 17. SAVE ALL RESULTS
# ══════════════════════════════════════════════════════════════════════════════
results = {
    "backend"           : backend.name,
    "shots"             : SHOTS,
    "n_test_samples"    : int(len(all_labels)),
    "circuit_info"      : gate_info,
    "physical_qubits"   : physical_qubits,
    "failed_batches"    : failed_batches,
    "simulator_metrics" : {k: round(v, 4) for k, v in sim_m.items()},
    "hardware_metrics"  : {k: round(v, 4) for k, v in hw_m.items()},
    "delta"             : {k: round(hw_m[k] - sim_m[k], 4) for k in sim_m},
    "elapsed_min"       : round(elapsed / 60, 2),
    "expectation_value_stats": {
        f"Q{i}": {
            "sim_mean": round(float(sim_evs[:, i].mean()), 4),
            "sim_std" : round(float(sim_evs[:, i].std()),  4),
            "hw_mean" : round(float(hw_evs[:, i].mean()),  4),
            "hw_std"  : round(float(hw_evs[:, i].std()),   4),
            "shift"   : round(float(hw_evs[:, i].mean() - sim_evs[:, i].mean()), 4),
        }
        for i in range(N_QUBITS)
    }
}
with open(os.path.join(OUT_DIR, "hardware_results.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"\n{'='*55}")
print("  HARDWARE EVALUATION COMPLETE")
print(f"{'='*55}")
print(f"  Simulator accuracy : {sim_m['accuracy']:.2f}%")
print(f"  Hardware  accuracy : {hw_m['accuracy']:.2f}%")
print(f"  Accuracy drop      : {hw_m['accuracy'] - sim_m['accuracy']:.2f}%")
print(f"  Elapsed            : {elapsed/60:.1f} min")
if failed_batches:
    print(f"\n  ⚠️  {len(failed_batches)} batch(es) stored as zeros: {failed_batches}")
    print(f"  Re-run the script to retry only those batches.")
print(f"\n  Output files in: {OUT_DIR}/")

Avoid re-runnign the failed batch jobs; the above code is self sufficient to run automaticallly for the lost one--- 
If you want to know what is remaining, then go through the code below :)

In [ ]:
import numpy as np, json, os

OUT_DIR = "/kaggle/working/hardware_eval"

# Load what was saved
hw_evs_partial = np.load(os.path.join(OUT_DIR, "hw_evs_partial.npy"))
all_labels      = np.load(os.path.join(OUT_DIR, "test_labels.npy"))

print(f"Partial EVs shape : {hw_evs_partial.shape}")
print(f"Total samples     : {len(all_labels)}")
print(f"Batches recovered : {hw_evs_partial.shape[0] // 10}")

# Check which rows are all zeros (failed batches)
zero_rows = np.where((hw_evs_partial == 0).all(axis=1))[0]
print(f"\nZero rows (failed samples): {zero_rows.tolist()}")
print(f"Failed batch indices      : {sorted(set(r // 10 for r in zero_rows))}")

In [ ]:
import numpy as np, json, os

OUT_DIR = "/kaggle/working/hardware_eval"

# Load the 190-row partial
old = np.load(os.path.join(OUT_DIR, "hw_evs_partial.npy"))  # (190, 4)

# Pad to full 300 rows
full = np.zeros((300, 4))
full[:old.shape[0]] = old
np.save(os.path.join(OUT_DIR, "hw_evs_partial.npy"), full)

# Mark batches 0–18 as done
batches_done = list(range(19))
with open(os.path.join(OUT_DIR, "batches_done.json"), "w") as f:
    json.dump(batches_done, f)

print("✅ Fixed — batches 0–18 marked done, array padded to (300, 4)")
print(f"Pending batches: 19–29 (11 batches remaining)")